# Week 1 — Solutions
## Distributions and honest reporting

Reference solutions for `exercises/01-distributions.ipynb`. These are one defensible
answer, not the only one. Yours may be better.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("../../assets/mplstyle/course.mplstyle")
df = pd.read_csv("../data/llm_eval_results.csv")


## Solution 1 — Cost vs. accuracy on `mmlu`

In [ ]:
mmlu = df[df["benchmark"] == "mmlu"]
agg = (mmlu.groupby(["model", "family", "params_b"], as_index=False)
            .agg(accuracy=("accuracy", "mean"),
                 cost=("cost_per_1k_tokens", "mean")))

fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=agg, x="cost", y="accuracy", hue="family",
                size="params_b", sizes=(40, 240), alpha=0.8, ax=ax)
ax.set_xscale("log")
ax.axhline(0.70, color="grey", lw=0.8, ls="--")
ax.text(agg["cost"].min(), 0.71, "70 % threshold", color="grey", fontsize=9)
ax.set_xlabel("USD / 1k tokens (log)")
ax.set_ylabel("mean accuracy on mmlu")
ax.set_title("Cost vs. accuracy on mmlu")
sns.move_legend(ax, "upper left", bbox_to_anchor=(1.02, 1))
plt.show()

# Cheapest model above 70%:
above = agg[agg["accuracy"] >= 0.70].sort_values("cost")
above.head(3)


**Reading.** The cheapest model that clears 70 % on `mmlu` is the smallest one
that the scaling line crosses the threshold for; reading off the chart, that is typically
a mid-size open-weights model rather than the frontier API model. The leaderboard table
does not show this — it ranks by accuracy, so the cheapest acceptable option is invisible.

## Solution 2 — Honest leaderboard

In [ ]:
leaderboard = (df.groupby(["model", "benchmark"])
                  .accuracy.mean().unstack("benchmark").round(3))
leaderboard.head(8)


In [ ]:
g = sns.catplot(
    data=df, kind="strip", x="model", y="accuracy", col="benchmark",
    col_wrap=4, height=2.8, aspect=1.4, alpha=0.7, jitter=0.18,
)
for ax in g.axes.flat:
    ax.tick_params(axis="x", rotation=60)
    for lab in ax.get_xticklabels():
        lab.set_horizontalalignment("right")
plt.show()


**Reading.** Models near the bottom of the scale on hard benchmarks (e.g. `math`,
`gsm8k`) show very tight spreads near zero — they are reliably bad. Mid-tier models
often show the widest spreads, indicating that the benchmark is right at the edge of
their capability. Frontier models cluster tightly at the top. A high-variance,
high-mean model is the most interesting from a methods-comparison standpoint and the
easiest to mis-report.

## Solution 3 — Misleading vs. honest

In [ ]:
# Pick two close models on a single benchmark
sub = df[(df["benchmark"] == "mmlu") & (df["model"].isin(["llama-13b", "mistral-7b"]))]
means = sub.groupby("model").accuracy.mean()

# Misleading: truncated y, narrow aspect, red-vs-grey
fig, ax = plt.subplots(figsize=(3, 4))
ax.bar(means.index, means.values, color=["#888888", "#D62728"])
ax.set_ylim(means.min() - 0.005, means.max() + 0.005)
ax.set_title("OURS WINS")
ax.set_ylabel("accuracy")
plt.tight_layout()
plt.savefig("../figures/misleading.png", dpi=150)
plt.show()

# Honest: full y, neutral colours, raw points overlaid
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=sub, x="model", y="accuracy", color="#999999",
            errorbar=("ci", 95), ax=ax)
sns.stripplot(data=sub, x="model", y="accuracy", color="#0072B2",
              alpha=0.8, ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Mean accuracy on mmlu (95 % CI, raw seeds overlaid)")
plt.tight_layout()
plt.savefig("../figures/honest.png", dpi=150)
plt.show()


**Reading.** The misleading version is a small chart with a 3-pixel-tall bar, no
visible variance, and a red-vs-grey palette that reads as "ours / theirs". The fix
restores the full y-axis, shows the raw seeds, and uses a neutral palette. The same
data tells two different stories.
